# Import the required libraries

All the librairies in requirements.txt
- pip install -r requirements.txt

In [110]:
import requests
import json
import dotenv
import os
import time
from oauthlib.oauth2 import BackendApplicationClient
from oauthlib.oauth2 import TokenExpiredError
from requests_oauthlib import OAuth2Session
from requests.auth import HTTPBasicAuth

# Load the SSE API Key and Secret Key
- API and Secret Key generated on SSE Admin Page
- load values in ".env" file

In [111]:
token_url = os.environ.get('TOKEN_URL') or 'https://api.sse.cisco.com/auth/v2/token'

# Export/Set the environment variables
dotenv.load_dotenv('../.env')
client_id = os.environ.get('API_KEY')
client_secret = os.environ.get('API_SECRET')


# Generate the token required for authorization on API

In [112]:
class SecureAccessAPI:
    def __init__(self, url, ident, secret):
        self.url = url
        self.ident = ident
        self.secret = secret
        self.token = None

    def GetToken(self):
        auth = HTTPBasicAuth(self.ident, self.secret)
        client = BackendApplicationClient(client_id=self.ident)
        oauth = OAuth2Session(client=client)
        self.token = oauth.fetch_token(token_url=self.url, auth=auth)
        return self.token

In [113]:
for var in ['API_SECRET', 'API_KEY']:
    if os.environ.get(var) == None:
        print("Required environment variable: {} not set".format(var))
        exit()

# Get token
api = SecureAccessAPI(token_url, client_id, client_secret)
#print("Token: " + str(api.GetToken()))

token = api.GetToken()['access_token']
#print(token)

In [ ]:
headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/json",
}

url = "https://api.sse.cisco.com/reports/v2/identities/1346149008"
params = {
    "limit": 5000,
    "offset": 1 
    # Filter to AD groups only. Remove this line first run to discover
    # what `type.type` strings your org actually returns.
    #"identitytypes": "ad_groups",
    # Optional: narrow by name
    # "search": "UdeM",
}
response = requests.get(url, headers=headers, params=params)
print("Status:", response.status_code)
print(response.text.encode('utf8'))
#data = response.json()

# Extract Group IDs and save it to JSON format

In [ ]:
headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/json",
}

url = "https://api.sse.cisco.com/reports/v2/identities"
params = {
    "limit": 5000,
    "offset": 1 
    # Filter to AD groups only. Remove this line first run to discover
    # what `type.type` strings your org actually returns.
    #"identitytypes": "ad_groups",
    # Optional: narrow by name
    # "search": "UdeM",
}
response = requests.get(url, headers=headers, params=params)

# Ensure the directory "data" exists
#os.makedirs("data", exist_ok=True)

# Save the response as JSON to data/groupID.json
with open("data/groupID.json", "w", encoding="utf-8") as f:
    json.dump(response.json(), f, ensure_ascii=False, indent=4)
print(response.text.encode('utf8'))

# Extract Private-Resources ID and Save it to JSON format

In [ ]:
url = "https://api.sse.cisco.com/policies/v2/privateResources"
params = {
    "limit": 1000,
    # Filter to AD groups only. Remove this line first run to discover
    # what `type.type` strings your org actually returns.
    #"identitytypes": "ad_groups",
    # Optional: narrow by name
    # "search": "UdeM",
}

response = requests.get(url, headers=headers, params=params)

with open("data/private-resourceID.json", "w", encoding="utf-8") as f:
    json.dump(response.json(), f, ensure_ascii=False, indent=4)
print(response.text.encode('utf8'))

In [98]:
headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/json",
}


url = "https://api.sse.cisco.com/policies/v2/rules/2587073"
params = {
    "limit": 10,
    # Filter to AD groups only. Remove this line first run to discover
    # what `type.type` strings your org actually returns.
    #"identitytypes": "ad_groups",
    # Optional: narrow by name
    # "search": "UdeM",
}

response = requests.get(url, headers=headers, params=params)
print(response.text.encode('utf8'))

b'{"ruleId": 2587073, "ruleExternalId": null, "modifiedAt": "2026-05-04T16:59:18+00:00", "modifiedBy": "org/8167900/user/7118261", "ruleDescription": "", "ruleIName": null, "ruleSettings": [{"settingValue": "LOG_ALL", "settingId": 5, "settingName": "umbrella.logLevel"}, {"settingValue": "PRIVATE_NETWORK", "settingId": 9, "settingName": "umbrella.default.traffic"}], "ruleAction": "allow", "createdAt": "2026-05-04T16:59:18+00:00", "ruleName": "UdeM-PAR-psychia-carnin-he", "ruleIsDefault": false, "ruleConditions": [{"attributeId": 2, "attributeValue": [1346149008], "attributeName": "umbrella.source.identity_ids", "attributeOperator": "INTERSECT"}, {"attributeId": 33, "attributeValue": [525015], "attributeName": "umbrella.destination.private_resource_ids", "attributeOperator": "IN"}], "rulePriority": 113, "ruleIsEnabled": true, "ruleAccess": "private_network"}'


# Create a Single Access-Policy Rule with Static Variables

In [ ]:
url = "https://api.sse.cisco.com/policies/v2/rules"

headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/json",
    "Content-Type": "application/json"
}


payload = '''{
    "ruleName": "UdeM-API-Access-Rule-1",
    "ruleDescription": "UdeM-API-Rule-1-Private-Network",
    "ruleAction": "allow",
    "rulePriority": 1,
    "ruleIsEnabled": true,
    "ruleAccess": "private_network",
    "ruleConditions": [
        {
            "attributeName": "umbrella.source.identity_ids",
            "attributeValue": [1346149008],
            "attributeOperator": "INTERSECT"
        },
        {
            "attributeValue": [525015],
            "attributeName": "umbrella.destination.private_resource_ids",
            "attributeOperator": "IN"
    }
    ],
    "ruleSettings": [
        {
            "settingName": "umbrella.logLevel",
            "settingValue": "LOG_ALL"
        },
        {
            "settingValue": "PRIVATE_NETWORK",
            "settingName": "umbrella.default.traffic"
    }
    ]
 }'''

response = requests.request('POST', url, headers=headers, data = payload)
print(response.text.encode('utf8'))
print(response.status_code)


# Create Multiple Access-Policy Rules

In [122]:
import json
import requests
import time

# 1. Load the input data files
with open('sample-5-private-resources.json', 'r', encoding='utf-8') as f:
    sample_rules = json.load(f)

with open('data/groupID.json', 'r', encoding='utf-8') as f:
    groups_data = json.load(f)

with open('data/private-resourceID.json', 'r', encoding='utf-8') as f:
    resources_data = json.load(f)

# 2. Create lookup dictionaries for fast mapping
# Map AD Group 'label' to 'id' from groupID.json
group_lookup = {group['label']: group['id'] for group in groups_data.get('data', [])}

# Map Private Resource 'name' to 'resourceId' from private-resourceID.json
resource_lookup = {res['name']: res['resourceId'] for res in resources_data.get('items', [])}

url = "https://api.sse.cisco.com/policies/v2/rules"

# 3. Iterate over the sample rules and execute the API POST calls
for item in sample_rules:
    rule_name = item.get("Rule")
    rule_description = item.get("Description", "")
    ad_group_name = item.get("AD Group")
    private_res_name = item.get("Private Resource")
    
    # Extract Group ID (matches "AD Group" from sample to "label" in groupID.json)
    group_id = group_lookup.get(ad_group_name)
    
    # Extract Private Resource ID
    # Attempts to match the "Rule" name first. If not found, falls back to the "Private Resource" name.
    resource_id = resource_lookup.get(rule_name)
    if not resource_id:
        resource_id = resource_lookup.get(private_res_name)
    
    # Safety Check: Skip the rule if the Group ID or Resource ID could not be found
    if not group_id:
        print(f"[-] Skipping '{rule_name}': Could not find AD Group ID for '{ad_group_name}'")
        continue
    if not resource_id:
        print(f"[-] Skipping '{rule_name}': Could not find Private Resource ID matching '{rule_name}' or '{private_res_name}'")
        continue
        
    print(f"[*] Creating rule: {rule_name}...")
    
    # Construct the payload dynamically
    payload = {
        "ruleName": rule_name,
        "ruleDescription": rule_description,
        "ruleAction": "allow",
        "rulePriority": 1,
        "ruleIsEnabled": True,
        "ruleAccess": "private_network",
        "ruleConditions": [
            {
                "attributeName": "umbrella.source.identity_ids",
                "attributeValue": [group_id],
                "attributeOperator": "INTERSECT"
            },
            {
                "attributeName": "umbrella.destination.private_resource_ids",
                "attributeValue": [resource_id],
                "attributeOperator": "IN"
            }
        ],
        "ruleSettings": [
            {
                "settingName": "umbrella.logLevel",
                "settingValue": "LOG_ALL"
            },
            {
                "settingName": "umbrella.default.traffic",
                "settingValue": "PRIVATE_NETWORK"
            }
        ]
    }
    
    # Send the API request
    response = requests.post(url, headers=headers, json=payload)
    
    print(f"    Status Code: {response.status_code}")
    if response.status_code in [200, 201]:
        print(f"    [+] Successfully created rule!\n")
    else:
        print(f"    [-] Failed: {response.text}\n")
    #print(payload)    
    # Prevent hitting rate limits (429 errors)
    time.sleep(1)


[*] Creating rule: UdeM-PAR-psychia-carnin-he...
    Status Code: 400
    [-] Failed: {"msg":"Rule with the name  as UdeM-PAR-psychia-carnin-he already exists. Please provide a different name"}


[*] Creating rule: UdeM-PAR-regis-pr2sql2008...
    Status Code: 200
    [+] Successfully created rule!

[*] Creating rule: UdeM-PAR-cepsum-dvsport...
    Status Code: 200
    [+] Successfully created rule!

[*] Creating rule: UdeM-PAR-ti-novipro...
    Status Code: 200
    [+] Successfully created rule!

[*] Creating rule: UdeM-PAR-fas-alphonse...
    Status Code: 200
    [+] Successfully created rule!

